In [19]:
MODEL = "gemini-2.0-flash"
# MODEL = "gpt-4o"

In [20]:
from pathlib import Path
import pandas as pd

# Cell 0: load all CSV files in a directory into a dict of DataFrames

def load_all_csv(path='.', recursive=False, show_progress=True, **read_csv_kwargs):
    """
    Load all .csv files from `path` into a dict of pandas.DataFrame objects.
    - path: directory path (str or Path)
    - recursive: if True, search subdirectories
    - show_progress: print loading status
    - read_csv_kwargs: passed to pandas.read_csv
    Returns: dict mapping filename (with extension) -> DataFrame
    """
    p = Path(path)
    pattern = '**/*.csv' if recursive else '*.csv'
    files = sorted(p.glob(pattern))
    dfs = {}
    for f in files:
        if show_progress:
            print(f"Reading {f} ...", end=' ')
        try:
            # Primary attempt: default pandas inference
            df = pd.read_csv(f, **read_csv_kwargs)
        except Exception:
            # Fallbacks: try to auto-detect delimiter and common encodings
            tried = False
            for enc in ('utf-8', 'latin1'):
                try:
                    df = pd.read_csv(f, sep=None, engine='python', encoding=enc, **read_csv_kwargs)
                    tried = True
                    break
                except Exception:
                    continue
            if not tried:
                if show_progress:
                    print("failed")
                print(f"Failed to read {f!s}; skipping.")
                continue
        dfs[f.name] = df
        if show_progress:
            print(f"ok (shape={df.shape})")
    if show_progress:
        print(f"Loaded {len(dfs)} CSV files from {p.resolve()}")
    return dfs

# Example usage:
# dfs = load_all_csv('.', recursive=True)
# Access a DataFrame by filename: dfs['my_file.csv']

In [21]:
# combine csv files into one dataframe with columns ['q_id',"source_lang", "target_lang", q_src,q_tgt,a_src,a_tgt,correct_target]
def combine_csv_files(dfs):
    combined_df = pd.DataFrame()
    for filename, df in dfs.items():
        if all(col in df.columns for col in ['q_id', 'source_lang', 'target_lang', 'q_src', 'q_tgt', 'a_src', 'a_tgt', 'correct_target',"c_src",'gold_answer_src','c_tgt','gold_answer_tgt']):
            combined_df = pd.concat([combined_df, df[['q_id', 'source_lang', 'target_lang', 'q_src', 'q_tgt', 'a_src', 'a_tgt', 'correct_target',"c_src",'gold_answer_src','c_tgt','gold_answer_tgt']]], ignore_index=True)
        else:
            print(f"Skipping {filename}: missing required columns.")
    return combined_df

#TODO: specify your path here!!!!!!!!!! Models evaluations directory

path_to_evaluations = f'../../eval/artifacts/{MODEL}' # specify your directory here

all_eval_dfs = load_all_csv(path_to_evaluations, recursive=True)
print(f"Total files loaded: {len(all_eval_dfs)}")

combined_eval_df = combine_csv_files(all_eval_dfs) 
print(f"Combined DataFrame shape: {combined_eval_df.shape}")
combined_eval_df.head()

Reading ../../eval/artifacts/gemini-2.0-flash/all_metrics_summary.csv ... ok (shape=(11, 4))
Reading ../../eval/artifacts/gemini-2.0-flash/de_predictions.csv ... ok (shape=(39, 17))
Reading ../../eval/artifacts/gemini-2.0-flash/en_predictions.csv ... ok (shape=(39, 17))
Reading ../../eval/artifacts/gemini-2.0-flash/en_source_answers.csv ... ok (shape=(39, 4))
Reading ../../eval/artifacts/gemini-2.0-flash/es_predictions.csv ... ok (shape=(39, 17))
Reading ../../eval/artifacts/gemini-2.0-flash/fr_predictions.csv ... ok (shape=(39, 17))
Reading ../../eval/artifacts/gemini-2.0-flash/he_predictions.csv ... ok (shape=(39, 17))
Reading ../../eval/artifacts/gemini-2.0-flash/hi_predictions.csv ... ok (shape=(39, 17))
Reading ../../eval/artifacts/gemini-2.0-flash/id_predictions.csv ... ok (shape=(39, 17))
Reading ../../eval/artifacts/gemini-2.0-flash/it_predictions.csv ... ok (shape=(39, 17))
Reading ../../eval/artifacts/gemini-2.0-flash/ja_predictions.csv ... ok (shape=(39, 17))
Reading ../../e

,q_id,source_lang,target_lang,q_src,q_tgt,a_src,a_tgt,correct_target,c_src,gold_answer_src,c_tgt,gold_answer_tgt
0,33,en,de,In which country was AFN Bremerhaven located?,In welchem Land befand sich AFN Bremerhaven?,AFN Bremerhaven was located in **Germany**.,AFN Bremerhaven befand sich in Deutschland. Es...,True,"AFN Bremerhaven was originally an ""Armed Force...",Germany,AFN Bremerhaven war ursprünglich ein Sender de...,Deutschland
1,30,en,de,What was the position of August Joseph Donatel...,Welche Position hatte August Joseph Donatelli ...,August Joseph Donatelli was flying a B-24 Libe...,August Joseph Donatelli war Sergeant im Hecksc...,False,"August Joseph Donatelli (August 22, 1914 – May...",Tailgunner,August Joseph Donatelli (22. August 1914 – 24....,Heckschütze
2,55,en,de,What was the television network that aired The...,Welcher Fernsehsender strahlte Die größte kana...,The television network that aired The Greatest...,Die größte kanadische Erfindung wurde auf CBC ...,False,The Greatest Canadian Invention is a spiritual...,CBC Television,Die größte kanadische Erfindung ist eine spiri...,CBC Television
3,39,en,de,Since which decade did bowl games start counti...,Ab welchem Jahrzehnt wurden Bowl-Spiele in die...,Bowl game statistics started counting toward s...,Bowl-Spiele wurden ab der Saison 2002 in die S...,True,The Cincinnati Bearcats football statistical l...,2000s,Die statistischen Spitzenreiter des Cincinnati...,2000er
4,63,en,de,What is the original name of Entebbe General H...,Wie lautete der ursprüngliche Name des Allgeme...,The original name of Entebbe General Hospital ...,Das Allgemeine Krankenhaus Entebbe hieß ursprü...,False,"Entebbe General Hospital, commonly known as En...",Entebbe Grade B Hospital,"Allgemeines Krankenhaus Entebbe, allgemein bek...",Entebbe-Krankenhaus der Stufe B


In [22]:
# load all features datasets
features_dfs = load_all_csv('.', recursive=False)
print(f"Total feature files loaded: {len(features_dfs)}")
for filename, df in features_dfs.items():
    print(f"{filename}: shape={df.shape}")

Reading eclektic_long_article_language_counts.csv ... ok (shape=(468, 4))
Reading eclektic_long_qa_topics.csv ... ok (shape=(468, 6))
Reading eclektic_long_subset.csv ... ok (shape=(468, 12))
Reading eclektic_long_subset_macro_features.csv ... ok (shape=(468, 24))
Reading eclektic_long_subset_with_question_type.csv ... ok (shape=(468, 13))
Reading eclektic_long_with_cooc_features.csv ... ok (shape=(468, 31))
Reading syntactic_complexity.csv ... ok (shape=(468, 20))
Reading training_data.csv ... ok (shape=(468, 36))
Reading training_data_missing_macro.csv ... ok (shape=(468, 26))
Loaded 9 CSV files from /Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/data/processed
Total feature files loaded: 9
eclektic_long_article_language_counts.csv: shape=(468, 4)
eclektic_long_qa_topics.csv: shape=(468, 6)
eclektic_long_subset.csv: shape=(468, 12)
eclektic_long_subset_macro_features.csv: shape=(468, 24)
eclektic_long_subset_with_question_type.csv: shape=(468, 13)
eclektic_long_

In [23]:
features_dfs["eclektic_long_article_language_counts.csv"].drop_duplicates(inplace=True)
temp_df = features_dfs["eclektic_long_article_language_counts.csv"][['q_id', 'language_version_count']]

merged_df = pd.merge(combined_eval_df, temp_df, on='q_id', how='left')
merged_df.shape




(468, 13)

In [24]:
features_dfs["eclektic_long_qa_topics.csv"].drop_duplicates(inplace=True)
temp_df = features_dfs["eclektic_long_qa_topics.csv"][['q_id', 'qa_topic']]
merged_df = pd.merge(merged_df, temp_df, on='q_id', how='left')

In [25]:
merged_df.shape


(468, 14)

In [26]:
temp_df = features_dfs["eclektic_long_subset_with_question_type.csv"][['q_id','question_type']]
temp_df.drop_duplicates(inplace=True)
merged_df = pd.merge(merged_df, temp_df, on='q_id', how='left')
merged_df.shape


/var/folders/ks/t4xykdjd3qj06ylqcn6zkprm0000gn/T/ipykernel_28883/1970492012.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_df.drop_duplicates(inplace=True)


(468, 15)

In [27]:
merged_df.head()

,q_id,source_lang,target_lang,q_src,q_tgt,a_src,a_tgt,correct_target,c_src,gold_answer_src,c_tgt,gold_answer_tgt,language_version_count,qa_topic,question_type
0,33,en,de,In which country was AFN Bremerhaven located?,In welchem Land befand sich AFN Bremerhaven?,AFN Bremerhaven was located in **Germany**.,AFN Bremerhaven befand sich in Deutschland. Es...,True,"AFN Bremerhaven was originally an ""Armed Force...",Germany,AFN Bremerhaven war ursprünglich ein Sender de...,Deutschland,1,geography,location
1,30,en,de,What was the position of August Joseph Donatel...,Welche Position hatte August Joseph Donatelli ...,August Joseph Donatelli was flying a B-24 Libe...,August Joseph Donatelli war Sergeant im Hecksc...,False,"August Joseph Donatelli (August 22, 1914 – May...",Tailgunner,August Joseph Donatelli (22. August 1914 – 24....,Heckschütze,1,history,event
2,55,en,de,What was the television network that aired The...,Welcher Fernsehsender strahlte Die größte kana...,The television network that aired The Greatest...,Die größte kanadische Erfindung wurde auf CBC ...,False,The Greatest Canadian Invention is a spiritual...,CBC Television,Die größte kanadische Erfindung ist eine spiri...,CBC Television,1,culture,organization
3,39,en,de,Since which decade did bowl games start counti...,Ab welchem Jahrzehnt wurden Bowl-Spiele in die...,Bowl game statistics started counting toward s...,Bowl-Spiele wurden ab der Saison 2002 in die S...,True,The Cincinnati Bearcats football statistical l...,2000s,Die statistischen Spitzenreiter des Cincinnati...,2000er,1,sports,date_time
4,63,en,de,What is the original name of Entebbe General H...,Wie lautete der ursprüngliche Name des Allgeme...,The original name of Entebbe General Hospital ...,Das Allgemeine Krankenhaus Entebbe hieß ursprü...,False,"Entebbe General Hospital, commonly known as En...",Entebbe Grade B Hospital,"Allgemeines Krankenhaus Entebbe, allgemein bek...",Entebbe-Krankenhaus der Stufe B,2,health,definition


In [28]:
merged_df = pd.merge(merged_df, features_dfs["eclektic_long_with_cooc_features.csv"], left_on=['q_id', 'target_lang'], right_on=['q_id', 'language'], how='left')
merged_df = merged_df.drop(columns=['question', 'answer', 'language'])
merged_df.head

<bound method NDFrame.head of      q_id source_lang target_lang  \
0      33          en          de   
1      30          en          de   
2      55          en          de   
3      39          en          de   
4      63          en          de   
..    ...         ...         ...   
463    52          en          zh   
464    54          en          zh   
465    50          en          zh   
466    58          en          zh   
467    48          en          zh   

                                                 q_src  \
0        In which country was AFN Bremerhaven located?   
1    What was the position of August Joseph Donatel...   
2    What was the television network that aired The...   
3    Since which decade did bowl games start counti...   
4    What is the original name of Entebbe General H...   
..                                                 ...   
463  What is the Planck mass denoted by in the equa...   
464  What month did the K-1: K’Festa 3 event take p...   
465

In [29]:
features_dfs["syntactic_complexity.csv"]
merged_df = pd.merge(merged_df, features_dfs["syntactic_complexity.csv"], left_on=['q_id', 'target_lang'], right_on=['q_id', 'language'], how='left').drop(columns=['language', 'original_lang','original_content', 'original_question', 'original_answer', 'content', 'question', 'answer', 'translated', 'title', 'url'])

In [30]:
merged_df.columns
merged_df.shape

(468, 50)

In [31]:

merged_df = pd.merge(merged_df, features_dfs["eclektic_long_subset_macro_features.csv"], left_on=['q_id', 'target_lang'], right_on=['q_id', 'language'], how='left').drop(columns=['language', 'original_lang','original_content', 'original_question', 'original_answer', 'content', 'question', 'answer', 'translated', 'title', 'url'])

In [32]:
merged_df.drop(columns=['source_lang_name', 'target_lang_name'], inplace=True)
merged_df.head()



,q_id,source_lang,target_lang,q_src,q_tgt,a_src,a_tgt,correct_target,c_src,gold_answer_src,...,source_family,source_genus,target_family,target_genus,source_script,source_syllables,source_wiki_size,target_script,target_syllables,target_wiki_size
0,33,en,de,In which country was AFN Bremerhaven located?,In welchem Land befand sich AFN Bremerhaven?,AFN Bremerhaven was located in **Germany**.,AFN Bremerhaven befand sich in Deutschland. Es...,True,"AFN Bremerhaven was originally an ""Armed Force...",Germany,...,Indo-European,Germanic,Indo-European,Germanic,LATIN,6949,7087876,LATIN,5100,3067614
1,30,en,de,What was the position of August Joseph Donatel...,Welche Position hatte August Joseph Donatelli ...,August Joseph Donatelli was flying a B-24 Libe...,August Joseph Donatelli war Sergeant im Hecksc...,False,"August Joseph Donatelli (August 22, 1914 – May...",Tailgunner,...,Indo-European,Germanic,Indo-European,Germanic,LATIN,6949,7087876,LATIN,5100,3067614
2,55,en,de,What was the television network that aired The...,Welcher Fernsehsender strahlte Die größte kana...,The television network that aired The Greatest...,Die größte kanadische Erfindung wurde auf CBC ...,False,The Greatest Canadian Invention is a spiritual...,CBC Television,...,Indo-European,Germanic,Indo-European,Germanic,LATIN,6949,7087876,LATIN,5100,3067614
3,39,en,de,Since which decade did bowl games start counti...,Ab welchem Jahrzehnt wurden Bowl-Spiele in die...,Bowl game statistics started counting toward s...,Bowl-Spiele wurden ab der Saison 2002 in die S...,True,The Cincinnati Bearcats football statistical l...,2000s,...,Indo-European,Germanic,Indo-European,Germanic,LATIN,6949,7087876,LATIN,5100,3067614
4,63,en,de,What is the original name of Entebbe General H...,Wie lautete der ursprüngliche Name des Allgeme...,The original name of Entebbe General Hospital ...,Das Allgemeine Krankenhaus Entebbe hieß ursprü...,False,"Entebbe General Hospital, commonly known as En...",Entebbe Grade B Hospital,...,Indo-European,Germanic,Indo-European,Germanic,LATIN,6949,7087876,LATIN,5100,3067614


In [33]:
merged_df.to_csv(f'../training_data/new/{MODEL}.csv', index=False)